In [ ]:
import json
import math
import re
import subprocess
import sys
import time
import warnings
from copy import deepcopy
from dataclasses import dataclass

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import torch
from gymnasium import spaces
from scipy import stats
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

## 1. Dependencies

In [ ]:
def install_dependencies():
    packages = [
        "transformers",
        "accelerate",
        "torch",
        "seaborn",
        "scipy",
        "pandas",
        "requests",
        "stable-baselines3",
        "shimmy",
        "gymnasium",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


try:
    from stable_baselines3 import PPO
except ImportError:
    install_dependencies()
    from stable_baselines3 import PPO

from transformers import AutoModelForCausalLM, AutoTokenizer

## 2. Language model setup

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = None
model = None

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        device_map="auto",
    )
    print(f"Loaded {MODEL_ID} on {DEVICE}.")
except Exception as exc:
    print(f"Model load skipped: {exc}")

## 3. Environment and data structures

In [ ]:
@dataclass
class CropProfile:
    name: str
    opt_min: float = 22.0
    opt_max: float = 25.0
    cost_heat: float = 5.0
    cost_vent: float = 1.0


@dataclass
class Telemetry:
    t_air: float
    rh: float
    t_sensor: float = 0.0


@dataclass
class Action:
    vent: float = 0.0
    heat: float = 0.0


PROFILE = CropProfile("Tomato")


class RealWeatherStation:
    def __init__(self, lat=23.8103, lon=90.4125):
        self.url = "https://archive-api.open-meteo.com/v1/archive"
        self.params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": "2024-01-01",
            "end_date": "2024-02-01",
            "hourly": ["temperature_2m", "shortwave_radiation"],
            "timezone": "Asia/Dhaka",
        }
        self.df = None

        try:
            response = requests.get(self.url, params=self.params, timeout=30)
            response.raise_for_status()
            data = response.json()
            self.df = pd.DataFrame(data["hourly"])
            self.df["solar_norm"] = self.df["shortwave_radiation"] / 1000.0
            print(f"Weather records loaded: {len(self.df)}")
        except Exception as exc:
            print(f"Weather API unavailable, using fallback weather: {exc}")

    def get_weather(self, step_index):
        if self.df is not None and not self.df.empty:
            idx = step_index % len(self.df)
            row = self.df.iloc[idx]
            return row["temperature_2m"], row["solar_norm"]

        hour = step_index % 24
        outside_temp = 20 + 5 * np.sin(hour / 24 * 2 * np.pi)
        solar = max(0, np.sin((hour - 6) / 12 * np.pi))
        return outside_temp, solar


class GreenhouseEnv:
    def __init__(self, seed=None):
        self.weather = RealWeatherStation()
        self.rng = np.random.default_rng(seed)
        self.vol = 500.0
        self.rho = 1.2
        self.cp = 1006.0
        self.thermal_mass = self.vol * self.rho * self.cp
        self.reset()

    def reset(self):
        self.state = Telemetry(t_air=20.0, rh=60.0, t_sensor=20.0)
        self.time = 0
        return self.state

    def step(self, action: Action):
        t_out, solar = self.weather.get_weather(self.time)
        steps = 60
        dt = 3600 / steps

        real_vent = np.clip(action.vent, 0.0, 1.0)
        real_heat = np.clip(action.heat, 0.0, 1.0)

        for _ in range(steps):
            q_sun = solar * 40.0
            vent_flow = 2.5 * math.sqrt(real_vent + 0.01) * self.rho
            q_vent = vent_flow * self.cp * (t_out - self.state.t_air)
            q_heat = real_heat * 40000.0
            q_trans = 5.0 * 200.0 * (t_out - self.state.t_air)
            q_net = q_sun + q_vent + q_heat + q_trans
            self.state.t_air += (q_net / self.thermal_mass) * dt

        self.time += 1
        self.state.t_sensor = self.state.t_air + self.rng.normal(0, 0.3)
        return deepcopy(self.state), t_out

## 4. Reinforcement-learning wrapper

In [ ]:
class GymGreenhouseWrapper(gym.Env):
    def __init__(self):
        super().__init__()
        self.env = GreenhouseEnv()
        self.action_space = spaces.Box(low=0, high=1, shape=(2,), dtype=np.float32)
        self.observation_space = spaces.Box(low=-10, high=50, shape=(2,), dtype=np.float32)

    def reset(self, seed=None, options=None):
        state = self.env.reset()
        t_out, _ = self.env.weather.get_weather(0)
        obs = np.array([state.t_sensor, t_out], dtype=np.float32)
        return obs, {}

    def step(self, action):
        state, t_out = self.env.step(Action(float(action[0]), float(action[1])))
        reward = -abs(state.t_sensor - 23.5) - (action[1] * 0.5 + action[0] * 0.1)
        obs = np.array([state.t_sensor, t_out], dtype=np.float32)
        return obs, reward, False, False, {}

## 5. Controllers

In [ ]:
class PIDAgent:
    def __init__(self):
        self.kp, self.ki, self.kd = 2.0, 0.1, 0.5
        self.prev_err = 0.0
        self.integral = 0.0

    def get_action(self, state, t_out):
        error = 23.5 - state.t_sensor
        self.integral += error
        derivative = error - self.prev_err
        output = self.kp * error + self.ki * self.integral + self.kd * derivative
        self.prev_err = error

        if output > 0:
            return Action(heat=np.clip(output, 0, 1), vent=0.0)
        return Action(heat=0.0, vent=np.clip(abs(output), 0, 1))


class FuzzyAgent:
    def __init__(self):
        self.prev_error = 0.0

    def _trimf(self, x, abc):
        a, b, c = abc
        if x <= a or x >= c:
            return 0.0
        if a < x <= b:
            return (x - a) / (b - a)
        if b < x < c:
            return (c - x) / (c - b)
        return 0.0

    def get_action(self, state, t_out):
        error = 23.5 - state.t_sensor
        self.prev_error = error

        mu_nb = self._trimf(error, [-10, -5, -1])
        mu_ns = self._trimf(error, [-3, -1, 0])
        mu_ps = self._trimf(error, [0, 1, 3])
        mu_pb = self._trimf(error, [1, 5, 10])

        heat_val = (mu_pb * 1.0 + mu_ps * 0.4) / (mu_pb + mu_ps + 1e-6)
        vent_val = (mu_nb * 1.0 + mu_ns * 0.4) / (mu_nb + mu_ns + 1e-6)

        if heat_val > vent_val:
            vent_val = 0.0
        else:
            heat_val = 0.0

        return Action(vent=float(vent_val), heat=float(heat_val))


class DRLAgent:
    def __init__(self):
        self.env = GymGreenhouseWrapper()
        self.model = PPO("MlpPolicy", self.env, verbose=0)
        self.model.learn(total_timesteps=5000)

    def get_action(self, state, t_out):
        action, _ = self.model.predict(np.array([state.t_sensor, t_out]))
        return Action(vent=float(action[0]), heat=float(action[1]))


def parse_llm_json(text):
    try:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            payload = json.loads(match.group(0))
            return payload.get("vent", 0.0), payload.get("heat", 0.0)
    except Exception:
        pass

    vent = 0.5 if "vent" in text.lower() and "1" in text else 0.0
    heat = 0.5 if "heat" in text.lower() and "1" in text else 0.0
    return vent, heat


class ShieldOnlyAgent:
    def get_action(self, state, t_out):
        if state.t_sensor > PROFILE.opt_max:
            return Action(vent=1.0, heat=0.0)
        if state.t_sensor < PROFILE.opt_min:
            return Action(vent=0.0, heat=1.0)
        if state.t_sensor > 24.0:
            return Action(vent=0.3, heat=0.0)
        if state.t_sensor < 23.0:
            return Action(vent=0.0, heat=0.3)
        return Action(0.0, 0.0)


class LLMOnlyAgent:
    def __init__(self):
        self.prev_action = Action(0.0, 0.0)

    def get_action(self, state, t_out):
        start_time = time.time()
        target_action = Action(0.0, 0.0)

        if model is not None and tokenizer is not None:
            prompt = (
                f"Greenhouse status: Temp={state.t_sensor:.1f}C (target 23.5C). "
                'Respond with JSON: {"vent": <0.0-1.0>, "heat": <0.0-1.0>}'
            )
            try:
                inputs = tokenizer([prompt], return_tensors="pt").to(DEVICE)
                with torch.no_grad():
                    output = model.generate(**inputs, max_new_tokens=20, do_sample=False)
                decoded = tokenizer.decode(output[0], skip_special_tokens=True)
                vent, heat = parse_llm_json(decoded)
                target_action = Action(vent, heat)
            except Exception:
                pass

        return target_action, time.time() - start_time


class GreenGuardFull:
    def __init__(self):
        self.prev_action = Action(0.0, 0.0)
        self.smoothed_temp = None

    def get_action(self, state, t_out):
        start_time = time.time()

        if self.smoothed_temp is None:
            self.smoothed_temp = state.t_sensor
        self.smoothed_temp = 0.2 * state.t_sensor + 0.8 * self.smoothed_temp

        target_action = Action(0.0, 0.0)
        if model is not None and tokenizer is not None:
            prompt = (
                f"Greenhouse status: Temp={self.smoothed_temp:.1f}C. "
                'Respond with JSON: {"vent": <float>, "heat": <float>}'
            )
            try:
                inputs = tokenizer([prompt], return_tensors="pt").to(DEVICE)
                with torch.no_grad():
                    output = model.generate(**inputs, max_new_tokens=20, do_sample=False)
                decoded = tokenizer.decode(output[0], skip_special_tokens=True)
                vent, heat = parse_llm_json(decoded)
                target_action = Action(vent, heat)
            except Exception:
                pass

        if self.smoothed_temp > PROFILE.opt_max:
            target_action = Action(1.0, 0.0)
        elif self.smoothed_temp < PROFILE.opt_min:
            target_action = Action(0.0, 1.0)

        slew = 0.2
        final_vent = np.clip(
            target_action.vent,
            self.prev_action.vent - slew,
            self.prev_action.vent + slew,
        )
        final_heat = np.clip(
            target_action.heat,
            self.prev_action.heat - slew,
            self.prev_action.heat + slew,
        )

        self.prev_action = Action(final_vent, final_heat)
        return self.prev_action, time.time() - start_time

## 6. Experiment loop

In [ ]:
def run_experiment():
    seeds = [42, 101, 777, 999, 123]
    agents = ["PID", "Fuzzy", "PPO", "Shield-Only", "LLM-Only", "GreenGuard"]
    records = []

    print(f"Running experiment across {len(seeds)} seeds.")

    for seed in seeds:
        print(f"Seed {seed}")
        active_agents = {
            "PID": PIDAgent(),
            "Fuzzy": FuzzyAgent(),
            "PPO": DRLAgent(),
            "Shield-Only": ShieldOnlyAgent(),
            "LLM-Only": LLMOnlyAgent(),
            "GreenGuard": GreenGuardFull(),
        }

        for name, agent in active_agents.items():
            env = GreenhouseEnv(seed=seed)
            total_rmse = 0.0
            total_cost = 0.0
            total_latency = 0.0
            violations = 0
            steps = 120

            for _ in range(steps):
                t_out, _ = env.weather.get_weather(env.time)

                if name in ["GreenGuard", "LLM-Only"]:
                    action, latency = agent.get_action(env.state, t_out)
                    total_latency += latency
                else:
                    start_time = time.time()
                    action = agent.get_action(env.state, t_out)
                    total_latency += time.time() - start_time

                next_state, _ = env.step(action)
                total_rmse += (next_state.t_air - 23.5) ** 2
                total_cost += action.heat * PROFILE.cost_heat + action.vent * PROFILE.cost_vent

                if next_state.t_air < PROFILE.opt_min or next_state.t_air > PROFILE.opt_max:
                    violations += 1

            records.append(
                {
                    "Agent": name,
                    "Seed": seed,
                    "RMSE": np.sqrt(total_rmse / steps),
                    "Cost": total_cost,
                    "Violations": violations,
                    "Latency (s)": total_latency / steps,
                }
            )

    return pd.DataFrame(records)

## 7. Summary analysis

In [ ]:
def analyze_results(df):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=180)
    metrics = ["RMSE", "Cost", "Violations"]

    for ax, metric in zip(axes, metrics):
        sns.boxplot(x="Agent", y=metric, data=df, ax=ax, palette="viridis")
        ax.set_title(f"{metric} distribution")
        ax.tick_params(axis="x", rotation=45)
        ax.grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig("results_summary.png", dpi=600, bbox_inches="tight")
    plt.show()

    print("Summary statistics")
    summary = df.groupby("Agent").agg(["mean", "std"])
    print(summary.to_markdown())

    print("\nT-tests against GreenGuard")
    gg_rmse = df[df["Agent"] == "GreenGuard"]["RMSE"]

    for other in [agent for agent in df["Agent"].unique() if agent != "GreenGuard"]:
        other_rmse = df[df["Agent"] == other]["RMSE"]
        _, p_value = stats.ttest_ind(gg_rmse, other_rmse)
        label = "significant" if p_value < 0.05 else "not significant"
        print(f"{other:12s} | p-value: {p_value:.5f} ({label})")

## 8. Visualization data

In [ ]:
def generate_viz_data():
    try:
        df = df_results.copy()
    except NameError:
        print("No cached results found. Running the experiment first.")
        df = run_experiment()

    snapshot_agents = ["PID", "LLM-Only", "GreenGuard"]
    history = {name: {"vent": [], "heat": [], "error": []} for name in snapshot_agents}
    env = GreenhouseEnv(seed=42)

    for name in snapshot_agents:
        if name == "PID":
            agent = PIDAgent()
        elif name == "LLM-Only":
            agent = LLMOnlyAgent()
        else:
            agent = GreenGuardFull()

        env.reset()
        for _ in range(24):
            t_out, _ = env.weather.get_weather(env.time)
            if name in ["GreenGuard", "LLM-Only"]:
                action, _ = agent.get_action(env.state, t_out)
            else:
                action = agent.get_action(env.state, t_out)
            next_state, _ = env.step(action)

            history[name]["vent"].append(action.vent)
            history[name]["heat"].append(action.heat)
            history[name]["error"].append(next_state.t_air - 23.5)

    return df, history

## 9. Dashboard figure

In [ ]:
def plot_dashboard(df, history):
    fig = plt.figure(figsize=(20, 12), dpi=180)
    gs = fig.add_gridspec(3, 3)

    colors = {
        "PID": "#8e44ad",
        "Fuzzy": "#3498db",
        "PPO": "#f39c12",
        "Shield-Only": "#95a5a6",
        "LLM-Only": "#e74c3c",
        "GreenGuard": "#27ae60",
    }

    ax_pareto = fig.add_subplot(gs[0, :2])
    summary = df.groupby("Agent")[["RMSE", "Cost"]].mean().reset_index()

    for agent in summary["Agent"]:
        row = summary[summary["Agent"] == agent]
        x = row["RMSE"]
        y = row["Cost"]
        ax_pareto.scatter(x, y, s=300, c=colors.get(agent, "black"), edgecolor="white", linewidth=2, label=agent)
        ax_pareto.scatter(x, y, s=1000, c=colors.get(agent, "black"), alpha=0.12)

    ax_pareto.annotate(
        "Optimal region",
        xy=(summary["RMSE"].min(), summary["Cost"].min()),
        xytext=(summary["RMSE"].min() + 2, summary["Cost"].min() - 20),
        arrowprops=dict(facecolor="black", shrink=0.05),
        fontsize=12,
        fontweight="bold",
    )
    ax_pareto.set_title("A. Pareto frontier", loc="left", fontsize=14, fontweight="bold")
    ax_pareto.set_xlabel("RMSE")
    ax_pareto.set_ylabel("Energy cost")
    ax_pareto.grid(True, linestyle="--", alpha=0.4)
    ax_pareto.legend(loc="upper right", frameon=True)

    ax_radar = fig.add_subplot(gs[0, 2], polar=True)
    metrics = ["RMSE", "Cost", "Violations"]
    radar_data = df.groupby("Agent")[metrics].mean()
    radar_norm = pd.DataFrame(
        MinMaxScaler().fit_transform(radar_data),
        columns=metrics,
        index=radar_data.index,
    )
    angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
    angles += angles[:1]

    for agent in ["PID", "LLM-Only", "GreenGuard"]:
        values = radar_norm.loc[agent].tolist()
        values += values[:1]
        ax_radar.plot(angles, values, color=colors[agent], linewidth=2, label=agent)
        ax_radar.fill(angles, values, color=colors[agent], alpha=0.1)

    ax_radar.set_xticks(angles[:-1])
    ax_radar.set_xticklabels(metrics, fontsize=11, fontweight="bold")
    ax_radar.set_yticks([0.2, 0.4, 0.6, 0.8])
    ax_radar.set_yticklabels([])
    ax_radar.set_title("B. Normalized profile", fontsize=14, fontweight="bold", pad=20)

    ax_heat = fig.add_subplot(gs[1, :])
    heatmap_data = []
    y_labels = []
    for agent in ["PID", "LLM-Only", "GreenGuard"]:
        heatmap_data.append(history[agent]["vent"])
        y_labels.append(f"{agent} (Vent)")
        heatmap_data.append(history[agent]["heat"])
        y_labels.append(f"{agent} (Heat)")

    sns.heatmap(
        np.array(heatmap_data),
        ax=ax_heat,
        cmap="magma",
        cbar_kws={"label": "Actuator intensity"},
        linewidths=1,
        linecolor="white",
    )
    ax_heat.set_yticks(np.arange(len(y_labels)) + 0.5)
    ax_heat.set_yticklabels(y_labels, rotation=0, fontsize=10, fontweight="bold")
    ax_heat.set_xlabel("Hour")
    ax_heat.set_title("C. Actuator pattern over 24 hours", loc="left", fontsize=14, fontweight="bold")

    ax_ridge = fig.add_subplot(gs[2, :])
    subset_df = df[df["Agent"].isin(["PID", "Fuzzy", "PPO", "GreenGuard"])]
    sns.violinplot(data=subset_df, x="Agent", y="RMSE", ax=ax_ridge, palette=colors, inner="quartile", linewidth=1.5)
    sns.stripplot(data=subset_df, x="Agent", y="RMSE", ax=ax_ridge, color="white", size=4, jitter=True, edgecolor="black", linewidth=1)
    ax_ridge.set_title("D. Stability analysis", loc="left", fontsize=14, fontweight="bold")
    ax_ridge.set_ylabel("RMSE")
    ax_ridge.grid(True, axis="y", alpha=0.3)

    fig.tight_layout()
    fig.savefig("greenhouse_dashboard.png", dpi=600, bbox_inches="tight")
    plt.show()

## 10. Run the notebook

In [ ]:
df_results = run_experiment()
analyze_results(df_results)

data_df, data_hist = generate_viz_data()
plot_dashboard(data_df, data_hist)

In [ ]:
def plot_dashboard(df, history):
    fig = plt.figure(figsize=(20, 12), dpi=1200)
    gs = fig.add_gridspec(3, 3)

    colors = {
        "PID": "#8e44ad",
        "Fuzzy": "#3498db",
        "PPO": "#f39c12",
        "Shield-Only": "#95a5a6",
        "LLM-Only": "#e74c3c",
        "GreenGuard": "#27ae60",
    }

    ax_pareto = fig.add_subplot(gs[0, :2])
    summary = df.groupby("Agent")[["RMSE", "Cost"]].mean().reset_index()

    for agent in summary["Agent"]:
        row = summary[summary["Agent"] == agent]
        x = row["RMSE"]
        y = row["Cost"]
        ax_pareto.scatter(x, y, s=300, c=colors.get(agent, "black"), edgecolor="white", linewidth=2, label=agent)
        ax_pareto.scatter(x, y, s=1000, c=colors.get(agent, "black"), alpha=0.12)

    ax_pareto.annotate(
        "Optimal region",
        xy=(summary["RMSE"].min(), summary["Cost"].min()),
        xytext=(summary["RMSE"].min() + 2, summary["Cost"].min() - 20),
        arrowprops=dict(facecolor="black", shrink=0.05),
        fontsize=12,
        fontweight="bold",
    )
    ax_pareto.set_title("A. Pareto frontier", loc="left", fontsize=14, fontweight="bold")
    ax_pareto.set_xlabel("RMSE")
    ax_pareto.set_ylabel("Energy cost")
    ax_pareto.grid(True, linestyle="--", alpha=0.4)
    ax_pareto.legend(loc="upper right", frameon=True)

    ax_radar = fig.add_subplot(gs[0, 2], polar=True)
    metrics = ["RMSE", "Cost", "Violations"]
    radar_data = df.groupby("Agent")[metrics].mean()
    radar_norm = pd.DataFrame(
        MinMaxScaler().fit_transform(radar_data),
        columns=metrics,
        index=radar_data.index,
    )
    angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
    angles += angles[:1]

    for agent in ["PID", "LLM-Only", "GreenGuard"]:
        values = radar_norm.loc[agent].tolist()
        values += values[:1]
        ax_radar.plot(angles, values, color=colors[agent], linewidth=2, label=agent)
        ax_radar.fill(angles, values, color=colors[agent], alpha=0.1)

    ax_radar.set_xticks(angles[:-1])
    ax_radar.set_xticklabels(metrics, fontsize=11, fontweight="bold")
    ax_radar.set_yticks([0.2, 0.4, 0.6, 0.8])
    ax_radar.set_yticklabels([])
    ax_radar.set_title("B. Normalized profile", fontsize=14, fontweight="bold", pad=20)

    ax_heat = fig.add_subplot(gs[1, :])
    heatmap_data = []
    y_labels = []
    for agent in ["PID", "LLM-Only", "GreenGuard"]:
        heatmap_data.append(history[agent]["vent"])
        y_labels.append(f"{agent} (Vent)")
        heatmap_data.append(history[agent]["heat"])
        y_labels.append(f"{agent} (Heat)")

    sns.heatmap(
        np.array(heatmap_data),
        ax=ax_heat,
        cmap="magma",
        cbar_kws={"label": "Actuator intensity"},
        linewidths=1,
        linecolor="white",
    )
    ax_heat.set_yticks(np.arange(len(y_labels)) + 0.5)
    ax_heat.set_yticklabels(y_labels, rotation=0, fontsize=10, fontweight="bold")
    ax_heat.set_xlabel("Hour")
    ax_heat.set_title("C. Actuator pattern over 24 hours", loc="left", fontsize=14, fontweight="bold")

    ax_ridge = fig.add_subplot(gs[2, :])
    subset_df = df[df["Agent"].isin(["PID", "Fuzzy", "PPO", "GreenGuard"])]
    sns.violinplot(data=subset_df, x="Agent", y="RMSE", ax=ax_ridge, palette=colors, inner="quartile", linewidth=1.5)
    sns.stripplot(data=subset_df, x="Agent", y="RMSE", ax=ax_ridge, color="white", size=4, jitter=True, edgecolor="black", linewidth=1)
    ax_ridge.set_title("D. Stability analysis", loc="left", fontsize=14, fontweight="bold")
    ax_ridge.set_ylabel("RMSE")
    ax_ridge.grid(True, axis="y", alpha=0.3)

    fig.tight_layout()
    fig.savefig("greenhouse_dashboard.png", dpi=1200, bbox_inches="tight")
    plt.show()

In [ ]:
df_results = run_experiment()
analyze_results(df_results)

data_df, data_hist = generate_viz_data()
plot_dashboard(data_df, data_hist)